# Light Gradient Boosting Machine (LightGBM)

**Notebook 4 of 4, and optional** — the day does not depend on it. Gradient boosting again,
with two design decisions of its own: it grows its trees in a different order, and reads the
features through **histograms** rather than raw values. Both were made for large data, and
the first changes how you tune it.

In [ ]:
import time
import timeit

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

## 1. The data

`data/california_housing.csv`: 20,640 districts from the 1990 California census — one row is
a **district**. Six numbers describing its housing, the two coordinates, and `location`, a
four-value column one-hot encoded as the earlier notebooks encode their categorical columns.

The target `MedHouseVal` is the median house value in units of **$100,000**, so error is
RMSE in those units. It is a large table by this repository's standards, and that matters for
everything below.

In [ ]:
df = pd.read_csv("../data/california_housing.csv")

y = df["MedHouseVal"]
X = pd.get_dummies(df.drop(columns="MedHouseVal"), columns=["location"]).astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


def score(model, label):
    """Fit a model on the training districts, then report both errors and the fitting time.

    Args:
        model: any unfitted regressor with fit and predict.
        label: what to call it in the printed line.

    """
    start = time.perf_counter()
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - start

    train_rmse = root_mean_squared_error(y_train, model.predict(X_train))
    test_rmse = root_mean_squared_error(y_test, model.predict(X_test))
    print(f"{label:<34} train {train_rmse:.3f}   test {test_rmse:.3f}   {elapsed:5.2f}s")


print(f"{len(X_train)} training districts, {len(X_test)} test districts, {X.shape[1]} columns")

## 2. Which leaf to split next

XGBoost grows a tree **level-wise**: every node at one depth is split before any at the
next, so the tree is balanced and `max_depth` limits it naturally.

LightGBM grows **leaf-wise**, taking at each step the single split that reduces the loss
most across every leaf, wherever it sits. The tree ends up lopsided, and reaches a given
loss in fewer splits.

So the controls change: **`num_leaves`, not `max_depth`, is the size of a LightGBM tree**,
and its default of 31 would take a level-wise grower depth 5 to match.

In [ ]:
# verbose=-1 is near-compulsory: LightGBM otherwise reports on every fit
score(XGBRegressor(random_state=42, n_jobs=-1), "xgboost, defaults")
score(LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1), "lightgbm, defaults")

**A tie on the test districts, 0.458 each, in the same tenth of a second.** Untuned, neither
library is ahead of the other.

The training column is where they differ: **0.268 for XGBoost against 0.394 for LightGBM**.
The two reach the same test error, and XGBoost gets there having memorized far more of the
training districts. Its defaults grow 100 trees of depth 6; LightGBM's grow 100 trees of 31
leaves, and on a table this size that is the more restrained of the two.

## 3. `num_leaves`

Leaf-wise growth makes this the setting that decides everything. Too few leaves and the
tree cannot express the problem; too many and it chases individual districts, faster than a
level-wise grower would, since every split it makes is the most tempting available.

> **Exercise 1.** With `n_estimators=500`, `learning_rate=0.05` and `min_child_samples=5`,
> fit an `LGBMRegressor` at `num_leaves` of **15, 127 and 255**, and print training and test
> RMSE for each. Three sizes is enough: one too small, one about right, one too large. Which
> is which, and what is the training error doing across them?

<details><summary>
Click here for a hint...
</summary>

A small function returning a configured `LGBMRegressor` keeps the three calls to one line
each. Keep `verbose=-1` and `random_state=42` every time.

</details>

In [ ]:
def sized(leaves):
    """Build an LGBMRegressor of a given size, with everything else held fixed.

    Args:
        leaves: the num_leaves value to try.

    Returns:
        LGBMRegressor: unfitted, ready to hand to score().
    """
    return LGBMRegressor(
        num_leaves=leaves,
        n_estimators=500,
        learning_rate=0.05,
        min_child_samples=5,     # 20 would keep the largest trees from growing at all
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )


# three sizes rather than a sweep: the turn shows up in three fits as well as in ten
score(sized(15), "num_leaves=15")
score(sized(127), "num_leaves=127")
score(sized(255), "num_leaves=255")

<details><summary>
Click here to compare your answer...
</summary>

| `num_leaves` | train | test |
| --: | --: | --: |
| 15 | 0.375 | 0.458 |
| 127 | 0.130 | **0.438** |
| 255 | 0.058 | 0.454 |

The training column **never stops falling**, 0.375 to 0.058, while the test column turns
around after 127. That divergence is overfitting drawn in two columns, and it is why
`num_leaves` is the first thing to tune.

Past the turn the extra leaves are bought entirely out of the training set. Push further —
1,023 leaves — and training error reaches 0.002 while test climbs past 0.48.

Note where the turn sits. On a table of a few hundred rows it would arrive by 15 or 31; here
it takes 127, because the number of leaves a table can support scales with the rows in it.

</details>

## 4. Looking at features through histograms

Finding the best split of a continuous feature means trying thresholds: the exact method
sorts the values and tries each, at a cost proportional to the number of **rows**.

LightGBM bins each feature once before any tree is grown, into at most `max_bin` buckets,
255 by default. Every split search then runs over **bins instead of rows**, so its cost
stops depending on the row count. Thresholds become bin edges rather than any value in the
data — an approximation, so the question is what it costs in accuracy, and the obvious way
to answer it answers wrongly.

In [ ]:
def binned(bins):
    """Build the section 3 model with a given max_bin, everything else held fixed.

    Args:
        bins: the max_bin value to try.

    Returns:
        LGBMRegressor: unfitted, ready to hand to score().
    """
    return LGBMRegressor(
        max_bin=bins,
        # the size section 3 settled on, so only the binning changes
        num_leaves=127, n_estimators=500, learning_rate=0.05, min_child_samples=5,
        random_state=42, n_jobs=-1, verbose=-1,
    )


score(binned(255), "max_bin=255, the default")
score(binned(15), "max_bin=15")

**0.438 at the default 255 bins, 0.508 at 15.** Coarsening the thresholds costs accuracy
here, and the drop at 15 bins is a quarter of the way back to predicting the mean.

The training column moves with it, 0.130 to 0.224, so this is not overfitting being tamed —
the coarse model is simply worse at the problem. Sixteen thresholds cannot place a split
where the data wants one.

**`max_bin` is a speed knob, and the default is already the accurate end of it.** Leave it
alone unless fitting time is the problem, which is what section 5 measures.

## 5. Where the speed was supposed to be

Section 4 made a claim worth testing: binning makes the cost of a split search stop depending
on the row count. If that is true, the gap between LightGBM and a library that sorts rows
should open up as the table grows.

Two further mechanisms from Ke et al. (2017) are often credited for the speed, and neither is
doing the work below:

- **GOSS**, gradient-based one-side sampling: keep every row with a large gradient, sample
  from the rest, and reweight. It is **off unless you ask for it**, with `boosting_type="goss"`.
- **EFB**, exclusive feature bundling: pack features that are never non-zero together. It
  pays on wide sparse tables, and twelve columns is neither.

So the test below holds the columns fixed and changes only the number of rows, 500 trees
each, tree sizes matched.

In [ ]:
def race(X_part, y_part, label):
    """Time XGBoost against LightGBM on the same rows, with the tree sizes matched.

    Args:
        X_part: the feature rows to fit on.
        y_part: their targets.
        label: what to call this run in the printed line.

    """
    # depth 4 is 16 leaves at most, so the two libraries grow the same size of tree
    xgb = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4,
                       random_state=42, n_jobs=-1)
    lgb = LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=16,
                        min_child_samples=5, random_state=42, n_jobs=-1, verbose=-1)

    def median_fit(model):
        """Median of three timed fits, after one warm-up fit that is not timed."""
        model.fit(X_part, y_part)
        runs = timeit.repeat(lambda: model.fit(X_part, y_part), number=1, repeat=3)
        return sorted(runs)[1]

    xgb_time, lgb_time = median_fit(xgb), median_fit(lgb)
    print(f"{label:<22} xgboost {xgb_time:5.2f}s   lightgbm {lgb_time:5.2f}s   "
          f"lightgbm {xgb_time / lgb_time:.2f}x")


# the first 824 districts: the same columns, a fraction of the rows
race(X_train.iloc[:824], y_train.iloc[:824], "824 districts")

Run that cell twice and the number moves. At 824 rows both libraries finish in a fraction of
a second, which is well inside the range where process startup and thread scheduling matter
more than the split search does.

Now the same race, the same twelve columns, and twenty times the rows.

In [ ]:
race(X_train, y_train, f"{len(X_train)} districts")

**And it moves here too.** Twenty times the rows did not buy a gap that survives a rerun:
both libraries still fit in about a second, and across repeated runs either one can come out
ahead.

That is the result, and it is worth sitting with rather than explaining away. Everything in
this section removes a cost that **grows with the row count**, so it can only become visible
once that cost dominates the fixed work — process startup, thread scheduling, the per-tree
overhead each library pays regardless of table size. The paper's 20x is measured on tables of
millions of rows. At 16,512 the row-dependent cost is not the thing being measured.

The rule that follows: **choose LightGBM for speed when the table is large enough for speed
to be the problem.** If you cannot measure a difference, the table is not large enough for
the choice to matter — and the tables in this repository are not.

## Summary

- LightGBM grows **leaf-wise**, so its trees come out lopsided and **`num_leaves` replaces
  `max_depth`** as the size control.
- That size has to be matched to the table: the turn came at **127 leaves** on 16,512
  districts, and past it the training error kept falling while the test error climbed.
- **Histogram binning** makes a split search cost bins rather than rows, and coarsening it
  costs accuracy, so the default is the accurate end and `max_bin` is a speed knob.
- The 500-tree race could **not** measure what that knob buys: at 824 rows and at 16,512
  alike the two libraries finish within noise of each other. The saving is real, and it needs
  tables far larger than these before it can be seen.